In [1]:
from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import linregress
from scipy.stats import pearsonr

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

In [2]:
# ============================
# CONFIGURATION
# ============================
PHOTO_ROOT = Path("/Users/rebekahzhang/data/photometry")
PROCESSED_OUT = PHOTO_ROOT / "processed_output"
BEHAV_DIR = PHOTO_ROOT / "behav_analyzed"
MATCHED_SESSIONS_CSV = PHOTO_ROOT / "matched_sessions.csv"
BEHAV_LOG_CSV = BEHAV_DIR / "sessions_photometry_exp2.csv"
OUT_PLOTS_DIR = PHOTO_ROOT / "trial_plots"

PHOTOMETRY_LOG_URL = (
    "https://docs.google.com/spreadsheets/d/1B8KCnku1vQInKOFBuoLeuDND6CEUA4hT6qQ5zuhqvyU/"
    "export?format=csv&gid=1751471403"
)

SIGNAL_COL = "dff_zscored"  # or "dff_filtered"
GRAB_RESULTS_FILENAME = "linear_fit_results_GRAB_lstr.csv"
CHANNEL_LABEL = "GRAB (l str)"
NON_IMPULSIVE_CUTOFF = 0.5

PLOT_COLOR = "#27ae60"

In [3]:
# ============================
# DATASET OVERVIEW
# ============================
MOUSE_GROUP = {
    "RZ074": "long",
    "RZ075": "long",
    "RZ083": "short",
    "RZ085": "short",
    "RZ081": "long",
    "RZ082": "long",
}

_log = pd.read_csv(PHOTOMETRY_LOG_URL, parse_dates=["date"])
_log["group"] = _log["mouse"].map(MOUSE_GROUP).fillna("unassigned")

_sessions_per_mouse = (
    _log.groupby(["group", "mouse"])
    .agg(n_sessions=("date", "count"), date_range=("date", lambda x: f"{x.min().date()} – {x.max().date()}"))
    .reset_index()
    .sort_values(["group", "mouse"])
)

_sessions_per_group = (
    _log.groupby("group")
    .agg(n_mice=("mouse", "nunique"), n_sessions=("date", "count"))
    .reset_index()
    .sort_values("group")
)

print("=" * 55)
print("SESSIONS PER GROUP")
print("=" * 55)
print(_sessions_per_group.to_string(index=False))
print("=" * 55)
print("SESSIONS PER MOUSE")
print("=" * 55)
print(_sessions_per_mouse.to_string(index=False))


SESSIONS PER GROUP
group  n_mice  n_sessions
 long       4          42
short       2          26
SESSIONS PER MOUSE
group mouse  n_sessions              date_range
 long RZ074          12 2025-08-08 – 2025-08-28
 long RZ075          12 2025-08-08 – 2025-08-28
 long RZ081           9 2026-02-17 – 2026-03-05
 long RZ082           9 2026-02-17 – 2026-03-05
short RZ083          16 2026-01-13 – 2026-03-05
short RZ085          10 2026-01-13 – 2026-03-05


# DA ramp vs TW/DT

In [ ]:
def compute_trial_slopes(sig: pd.DataFrame, channel: str) -> pd.DataFrame:
    sig = sig.copy()
    sig["decision_time"] = sig["bg_length"] + sig["time_waited"]

    sig = sig.loc[sig["miss_trial"] == False].copy()
    sig = sig.loc[sig["channel"] == channel].copy()

    results: List[Dict[str, object]] = []
    for trial_num, trial_sig in sig.groupby("session_trial_num"):
        trial_data = trial_sig.iloc[0]
        decision_time = trial_data["decision_time"]

        sig_window = trial_sig[(trial_sig["trial_time"] >= 0) & (trial_sig["trial_time"] <= decision_time)]
        if len(sig_window) < 2:
            continue

        slope, intercept, r_value, p_value, std_err = linregress(
            sig_window["trial_time"],
            sig_window[SIGNAL_COL],
        )

        results.append(
            {
                "trial_num": int(trial_num),
                "slope": slope,
                "intercept": intercept,
                "r_squared": r_value**2,
                "p_value": p_value,
                "std_err": std_err,
                "bg_length": trial_data.get("bg_length", np.nan),
                "time_waited": trial_data.get("time_waited", np.nan),
                "decision_time": decision_time,
                "reward": trial_data.get("reward", np.nan),
                "n_points": len(sig_window),
            }
        )
    return pd.DataFrame(results)

def plot_trial_fit(
    trial_sig: pd.DataFrame,
    decision_time: float,
    slope: float,
    intercept: float,
    channel: str,
    save_path: Path,
) -> None:
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(
        trial_sig["trial_time"],
        trial_sig[SIGNAL_COL],
        label=channel,
        linewidth=1.4,
        alpha=0.85,
        color=PLOT_COLOR,
    )

    fit_x = np.array([0, decision_time])
    fit_y = slope * fit_x + intercept
    ax.plot(fit_x, fit_y, "--", linewidth=2, alpha=0.7, color=PLOT_COLOR, label="Fit")

    ax.axvline(0, color="green", linestyle="--", linewidth=2, alpha=0.7, label="Cue On")
    bg_length = float(trial_sig.iloc[0].get("bg_length", np.nan))
    if np.isfinite(bg_length):
        ax.axvline(bg_length, color="orange", linestyle="--", linewidth=2, alpha=0.7, label="BG End")
    ax.axvline(decision_time, color="blue", linestyle="--", linewidth=2, alpha=0.7, label="Decision")

    ax.set_xlabel("Trial Time (s)")
    ax.set_ylabel(SIGNAL_COL)
    ax.set_title(f"Trial {int(trial_sig.iloc[0]['session_trial_num'])}: {channel} Fit")
    ax.legend(loc="upper right", fontsize=9)
    ax.grid(alpha=0.3)
    plt.tight_layout()

    save_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)

def run_regression(df: pd.DataFrame, outcome_col: str = "decision_time") -> Dict[str, float]:
    # Clean data
    clean_df = df.dropna(subset=['slope', outcome_col])

    # Check if we have enough data
    if len(clean_df) < 2:
        return {
            "n": int(len(clean_df)),
            "coef": np.nan,
            "intercept": np.nan,
            "r_squared": np.nan,
            "r_value": np.nan,
            "p_value": np.nan,
            "std_err": np.nan,
        }

    # Check for variance in both variables (linregress fails if either has no variance)
    if clean_df['slope'].std() == 0 or clean_df[outcome_col].std() == 0:
        return {
            "n": int(len(clean_df)),
            "coef": np.nan,
            "intercept": np.nan,
            "r_squared": np.nan,
            "r_value": np.nan,
            "p_value": np.nan,
            "std_err": np.nan,
        }

    slope_coef, intercept, r_value, p_value, std_err = linregress(clean_df["slope"], clean_df[outcome_col])
    return {
        "n": int(len(clean_df)),
        "coef": slope_coef,
        "intercept": intercept,
        "r_squared": r_value**2,
        "r_value": r_value,
        "p_value": p_value,
        "std_err": std_err,
    }


In [ ]:
merged_log = pd.read_csv(PHOTO_ROOT / "sessions_dff_behav_merged.csv", index_col=0)
session_summaries: List[Dict[str, object]] = []

# Load photometry log for per-session GRAB channel lookup
_photo_log = pd.read_csv(PHOTOMETRY_LOG_URL)
_photo_log["date"] = pd.to_datetime(_photo_log["date"]).dt.strftime("%Y-%m-%d")

_FIBER_CHANNEL_MAP = {
    2: {1: ("R0", "G2"), 2: ("R1", "G3")},
    4: {1: ("R0", "G4"), 2: ("R1", "G5"), 3: ("R2", "G6"), 4: ("R3", "G7")},
}

def get_grab_green_channel(mouse: str, date: str) -> str | None:
    """Return the green channel recording left-striatum GRAB for this session, or None."""
    row = _photo_log[(_photo_log["mouse"] == mouse) & (_photo_log["date"] == date)]
    if row.empty:
        return None
    row = row.iloc[0]
    n_fibers = sum(
        1 for i in range(1, 5)
        if pd.notna(row.get(f"fiber_{i}_area")) and str(row.get(f"fiber_{i}_area")).strip() != ""
    )
    if n_fibers not in _FIBER_CHANNEL_MAP:
        return None
    for fiber_num, (r_ch, g_ch) in _FIBER_CHANNEL_MAP[n_fibers].items():
        side = str(row.get(f"fiber_{fiber_num}_side", "")).strip().lower()
        area = str(row.get(f"fiber_{fiber_num}_area", "")).strip().lower()
        sensor = str(row.get(f"fiber_{fiber_num}_sensor", "")).strip()
        if side == "l" and area in ("str", "striatum") and sensor.upper() == "GRAB":
            return g_ch
    return None

In [ ]:
# ============================
# SESSION & TRIAL SUMMARY
# ============================
MOUSE_GROUP = {
    "RZ074": "long",
    "RZ075": "long",
    "RZ083": "short",
    "RZ085": "short",
}

merged_log["group"] = merged_log["mouse"].map(MOUSE_GROUP)

# Count trials per session from behavior CSVs
trial_counts = []
for _, session_info in merged_log.iterrows():
    trials_path = BEHAV_DIR / str(session_info["dir"]) / f"trials_analyzed_{session_info['dir']}.csv"
    if not trials_path.exists():
        continue
    trials = pd.read_csv(trials_path, index_col=0)
    trial_counts.append({
        "group":      session_info["group"],
        "mouse":      session_info["mouse"],
        "session_id": session_info["session_id"],
        "n_trials":   len(trials),
    })

trial_df = pd.DataFrame(trial_counts)

mouse_summary = (
    trial_df.groupby(["group", "mouse"])
    .agg(n_sessions=("session_id", "nunique"), n_trials=("n_trials", "sum"))
    .reset_index()
)

group_summary = (
    trial_df.groupby("group")
    .agg(n_mice=("mouse", "nunique"), n_sessions=("session_id", "nunique"), n_trials=("n_trials", "sum"))
    .reset_index()
)

print("=" * 50)
print("SUMMARY BY MOUSE")
print("=" * 50)
print(mouse_summary.to_string(index=False))
print("\n" + "=" * 50)
print("SUMMARY BY GROUP")
print("=" * 50)
print(group_summary.to_string(index=False))

In [ ]:
for _, session_info in merged_log.iterrows():
    session_id = str(session_info["session_id"])

    sig_path = PROCESSED_OUT / session_id / "photometry_with_trial_data.csv"
    if not sig_path.exists():
        print(f"  Skipping {session_id}: missing photometry_with_trial_data.csv — run 3_fp_behavior_check_plots.py process() first")
        continue

    grab_ch = get_grab_green_channel(session_info["mouse"], session_info["date"])
    if grab_ch is None:
        print(f"  Skipping {session_id}: no left-striatum GRAB fiber in photometry log")
        continue

    results_path = PROCESSED_OUT / session_id / GRAB_RESULTS_FILENAME
    if results_path.exists():
        print(f"  Skipping {session_id} (slopes already computed)")
        continue

    print(f"Computing slopes: {session_id} (channel: {grab_ch})")
    sig = pd.read_csv(sig_path, dtype={"previous_trial_miss_trial": "object"})

    results_df = compute_trial_slopes(sig, grab_ch)
    if results_df.empty:
        print(f"  Skipping {session_id}: no valid trials for channel {grab_ch}")
        continue

    results_df.to_csv(results_path, index=False)

    # Plot each trial fit
    plot_dir = OUT_PLOTS_DIR / f"{session_info['mouse']}_{session_info['date']}_linear_fits"
    sig_filtered = sig.loc[(sig["miss_trial"] == False) & (sig["channel"] == grab_ch)].copy()

    for _, row in results_df.iterrows():
        tr = row["trial_num"]
        trial_sig = sig_filtered[sig_filtered["session_trial_num"] == tr]
        if trial_sig.empty:
            continue
        plot_trial_fit(
            trial_sig=trial_sig,
            decision_time=float(row["decision_time"]),
            slope=float(row["slope"]),
            intercept=float(row["intercept"]),
            channel=grab_ch,
            save_path=plot_dir / f"trial_{int(tr):03d}_linear_fit.png",
        )

print("Done computing slopes.")

In [ ]:
for _, session_info in merged_log.iterrows():
    session_id = str(session_info["session_id"])

    grab_ch = get_grab_green_channel(session_info["mouse"], session_info["date"])
    if grab_ch is None:
        print(f"  Skipping {session_id}: no left-striatum GRAB fiber in photometry log")
        continue

    lr_path = PROCESSED_OUT / session_id / GRAB_RESULTS_FILENAME
    if not lr_path.exists():
        print(f"  Skipping {session_id}: missing {GRAB_RESULTS_FILENAME}")
        continue

    lr = pd.read_csv(lr_path)
    if lr.empty:
        print(f"  Skipping {session_id}: empty fit results")
        continue

    behav_subdir = str(session_info["dir"])
    trials_path = BEHAV_DIR / behav_subdir / f"trials_analyzed_{behav_subdir}.csv"
    trials = pd.read_csv(trials_path, index_col=0)

    trials_with_slope = trials.merge(
        lr[['trial_num', 'slope', 'intercept', 'r_squared', 'p_value', 'std_err']],
        left_on='session_trial_num',
        right_on='trial_num',
        how='left'
    )
    trials_with_slope['decision_time'] = trials_with_slope['bg_length'] + trials_with_slope['time_waited']

    all_df = trials_with_slope
    non_impulsive_df = trials_with_slope[trials_with_slope["time_waited"] > NON_IMPULSIVE_CUTOFF]
    good_df = trials_with_slope[trials_with_slope["good_trial"] == True]

    for label, df in [
        ("all", all_df),
        ("non_impulsive", non_impulsive_df),
        ("good", good_df)
    ]:
        for time_col in ["decision_time", "time_waited"]:
            stats = run_regression(df, outcome_col=time_col)

            row = {
                "session_id": session_id,
                "mouse": session_info.get("mouse", ""),
                "date": session_info.get("date", ""),
                "trial_group": label,
                "time_correlated": time_col,
                "n": stats["n"],
                "coef": stats["coef"],
                "intercept": stats["intercept"],
                "r2": stats["r_squared"],
                "r": stats["r_value"],
                "p": stats["p_value"],
                "stderr": stats["std_err"],
            }
            session_summaries.append(row)

    print(f"Processed {session_id}: {len(lr)} trials")

    summary_df = pd.DataFrame(session_summaries)
    summary_df.to_csv(PROCESSED_OUT / "all_sessions_regression_summary_GRAB_lstr.csv", index=False)

In [ ]:
summary_df

In [ ]:
summary_df.loc[summary_df['p']<0.05]

# Correlation analysis: DA signal vs time waited/decision time

In [ ]:
def extract_signal_features(sig: pd.DataFrame, channel: str) -> pd.DataFrame:
    """Extract DA signal features for each trial."""
    sig = sig.copy()
    sig["decision_time"] = sig["bg_length"] + sig["time_waited"]

    sig = sig.loc[sig["miss_trial"] == False].copy()
    sig = sig.loc[sig["channel"] == channel].copy()

    results = []
    for trial_num, trial_sig in sig.groupby("session_trial_num"):
        trial_data = trial_sig.iloc[0]
        decision_time = trial_data["decision_time"]

        sig_window = trial_sig[(trial_sig["trial_time"] >= 0) & (trial_sig["trial_time"] <= decision_time)]

        if len(sig_window) < 2:
            continue

        mean_signal = sig_window[SIGNAL_COL].mean()
        peak_signal = sig_window[SIGNAL_COL].max()
        min_signal = sig_window[SIGNAL_COL].min()

        sig_at_decision = sig_window.iloc[-1][SIGNAL_COL]
        sig_at_cue = sig_window.iloc[0][SIGNAL_COL]

        results.append({
            "trial_num": int(trial_num),
            "mean_signal": mean_signal,
            "peak_signal": peak_signal,
            "min_signal": min_signal,
            "sig_at_decision": sig_at_decision,
            "sig_at_cue": sig_at_cue,
            "signal_range": peak_signal - min_signal,
            "bg_length": trial_data.get("bg_length", np.nan),
            "time_waited": trial_data.get("time_waited", np.nan),
            "decision_time": decision_time,
            "reward": trial_data.get("reward", np.nan),
        })

    return pd.DataFrame(results)

def compute_correlations(df: pd.DataFrame, features: List[str], target_col: str) -> Dict[str, Dict]:
    """Compute Pearson correlation between each feature and target_col.
    Returns {feature: {'r': float, 'p': float, 'n': int}}.
    """
    results = {}
    for feat in features:
        clean = df[[feat, target_col]].dropna()
        n = len(clean)
        if n < 3 or clean[feat].std() == 0 or clean[target_col].std() == 0:
            results[feat] = {"r": np.nan, "p": np.nan, "n": n}
        else:
            r, p = pearsonr(clean[feat], clean[target_col])
            results[feat] = {"r": r, "p": p, "n": n}
    return results

In [ ]:
# Extract signal features for all sessions and pool into one dataframe
all_signal_features = []

for _, session_info in merged_log.iterrows():
    session_id = str(session_info["session_id"])
    sig_path = PROCESSED_OUT / session_id / "photometry_with_trial_data.csv"
    if not sig_path.exists():
        print(f"  Skipping {session_id}: missing photometry_with_trial_data.csv")
        continue

    grab_ch = get_grab_green_channel(session_info["mouse"], session_info["date"])
    if grab_ch is None:
        print(f"  Skipping {session_id}: no left-striatum GRAB fiber in photometry log")
        continue

    sig = pd.read_csv(sig_path, dtype={"previous_trial_miss_trial": "object"})
    feats = extract_signal_features(sig, grab_ch)
    if feats.empty:
        continue

    # Merge in per-trial slope from the already-computed fit results
    lr_path = PROCESSED_OUT / session_id / GRAB_RESULTS_FILENAME
    if lr_path.exists():
        lr = pd.read_csv(lr_path)[["trial_num", "slope"]]
        feats = feats.merge(lr, on="trial_num", how="left")

    feats["session_id"] = session_id
    feats["mouse"] = session_info.get("mouse", "")
    feats["date"] = session_info.get("date", "")
    all_signal_features.append(feats)

signal_features_all = pd.concat(all_signal_features, ignore_index=True) if all_signal_features else pd.DataFrame()
print(f"Pooled signal features: {len(signal_features_all)} trials across {len(all_signal_features)} sessions")
signal_features_all.head()

In [ ]:
# Compute correlations across all pooled sessions
features = ['mean_signal', 'peak_signal', 'min_signal', 'sig_at_decision',
            'sig_at_cue', 'signal_range', 'slope']

print(f"Pooled data: {len(signal_features_all)} trials across {signal_features_all['session_id'].nunique()} sessions\n")

print("=" * 60)
print("CORRELATIONS WITH TIME WAITED (all sessions pooled)")
print("=" * 60)
corr_time_waited = compute_correlations(signal_features_all, features, 'time_waited')
for feat, stats in corr_time_waited.items():
    sig_marker = "***" if stats['p'] < 0.001 else "**" if stats['p'] < 0.01 else "*" if stats['p'] < 0.05 else ""
    print(f"{feat:20s}: r={stats['r']:7.4f}, p={stats['p']:8.5f} {sig_marker} (n={stats['n']})")

print("\n" + "=" * 60)
print("CORRELATIONS WITH DECISION TIME (all sessions pooled)")
print("=" * 60)
corr_decision_time = compute_correlations(signal_features_all, features, 'decision_time')
for feat, stats in corr_decision_time.items():
    sig_marker = "***" if stats['p'] < 0.001 else "**" if stats['p'] < 0.01 else "*" if stats['p'] < 0.05 else ""
    print(f"{feat:20s}: r={stats['r']:7.4f}, p={stats['p']:8.5f} {sig_marker} (n={stats['n']})")

In [ ]:
# Visualize the correlations with scatter plots (pooled across all sessions)
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle(f'DA Signal Features vs Time Measures — {signal_features_all["session_id"].nunique()} sessions pooled ({CHANNEL_LABEL})', fontsize=14, fontweight='bold')

for idx, feature in enumerate(features[:3]):
    ax = axes[0, idx]
    clean_data = signal_features_all[[feature, 'time_waited']].dropna()
    if len(clean_data) > 0:
        ax.scatter(clean_data[feature], clean_data['time_waited'], alpha=0.2, s=15)
        if not np.isnan(corr_time_waited[feature]['r']):
            z = np.polyfit(clean_data[feature], clean_data['time_waited'], 1)
            x_line = np.linspace(clean_data[feature].min(), clean_data[feature].max(), 100)
            ax.plot(x_line, np.poly1d(z)(x_line), "r--", alpha=0.8, linewidth=2)
            r, pval = corr_time_waited[feature]['r'], corr_time_waited[feature]['p']
            ax.set_title(f'{feature}\nr={r:.3f}, p={pval:.4f}', fontsize=10)
    ax.set_xlabel(feature)
    ax.set_ylabel('Time Waited (s)')
    ax.grid(alpha=0.3)

for idx, feature in enumerate(features[:3]):
    ax = axes[1, idx]
    clean_data = signal_features_all[[feature, 'decision_time']].dropna()
    if len(clean_data) > 0:
        ax.scatter(clean_data[feature], clean_data['decision_time'], alpha=0.2, s=15, color='orange')
        if not np.isnan(corr_decision_time[feature]['r']):
            z = np.polyfit(clean_data[feature], clean_data['decision_time'], 1)
            x_line = np.linspace(clean_data[feature].min(), clean_data[feature].max(), 100)
            ax.plot(x_line, np.poly1d(z)(x_line), "r--", alpha=0.8, linewidth=2)
            r, pval = corr_decision_time[feature]['r'], corr_decision_time[feature]['p']
            ax.set_title(f'{feature}\nr={r:.3f}, p={pval:.4f}', fontsize=10)
    ax.set_xlabel(feature)
    ax.set_ylabel('Decision Time (s)')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Analyze: fitted DA value at decision time vs behavioral measures — pooled across all sessions
# DA_fitted_at_decision = slope * decision_time + intercept

all_trials_with_fit = []

for _, session_info in merged_log.iterrows():
    session_id = str(session_info["session_id"])
    lr_path = PROCESSED_OUT / session_id / GRAB_RESULTS_FILENAME
    trials_path = BEHAV_DIR / str(session_info["dir"]) / f"trials_analyzed_{session_info['dir']}.csv"

    if not lr_path.exists() or not trials_path.exists():
        continue

    lr = pd.read_csv(lr_path)
    if lr.empty:
        continue
    trials = pd.read_csv(trials_path, index_col=0)

    twf = trials.merge(
        lr[['trial_num', 'slope', 'intercept', 'r_squared']],
        left_on='session_trial_num', right_on='trial_num', how='left'
    )
    twf['decision_time'] = twf['bg_length'] + twf['time_waited']
    twf['da_at_decision'] = twf['slope'] * twf['decision_time'] + twf['intercept']
    twf['session_id'] = session_id
    all_trials_with_fit.append(twf)

trials_with_fit_all = pd.concat(all_trials_with_fit, ignore_index=True) if all_trials_with_fit else pd.DataFrame()
n_sessions = trials_with_fit_all['session_id'].nunique() if not trials_with_fit_all.empty else 0

print("=" * 70)
print(f"CORRELATION: Fitted DA value at decision time vs Behavioral measures")
print(f"({len(trials_with_fit_all)} trials across {n_sessions} sessions)")
print("=" * 70)

for time_col, label in [("time_waited", "TIME_WAITED"), ("decision_time", "DECISION_TIME")]:
    clean = trials_with_fit_all[['da_at_decision', time_col]].dropna()
    if len(clean) > 2 and clean['da_at_decision'].std() > 0 and clean[time_col].std() > 0:
        r, p = pearsonr(clean['da_at_decision'], clean[time_col])
        sig_marker = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
        print(f"\nDA at decision vs {label}:")
        print(f"  r = {r:7.4f}, p = {p:8.5f} {sig_marker}  (n={len(clean)})")
    else:
        print(f"\nDA at decision vs {label}: Insufficient data or no variance")

print("\n" + "=" * 70)

In [ ]:
# Visualize: fitted DA at decision time vs behavioral measures (pooled)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Fitted DA Value at Decision Time vs Behavioral Measures — {n_sessions} sessions pooled ({CHANNEL_LABEL})',
             fontsize=13, fontweight='bold')

for ax, time_col, color, ylabel in [
    (axes[0], 'time_waited',   'steelblue', 'Time Waited (s)'),
    (axes[1], 'decision_time', 'coral',     'Decision Time (s)'),
]:
    clean = trials_with_fit_all[['da_at_decision', time_col]].dropna()
    if len(clean) > 0:
        ax.scatter(clean[time_col], clean['da_at_decision'], alpha=0.3, s=20, color=color)
        if len(clean) > 2 and clean['da_at_decision'].std() > 0 and clean[time_col].std() > 0:
            z = np.polyfit(clean[time_col], clean['da_at_decision'], 1)
            x_line = np.linspace(clean[time_col].min(), clean[time_col].max(), 100)
            ax.plot(x_line, np.poly1d(z)(x_line), "r--", alpha=0.8, linewidth=2, label='Linear fit')
            r, p = pearsonr(clean['da_at_decision'], clean[time_col])
            sig_marker = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
            ax.set_title(f'DA at Decision vs {ylabel.split(" (")[0]}\nr={r:.3f}, p={p:.4f} {sig_marker}', fontsize=11)
            ax.legend()
    ax.set_xlabel(ylabel, fontsize=11)
    ax.set_ylabel('Fitted DA at Decision Time', fontsize=11)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Visualize remaining features (pooled across all sessions)
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle(f'Additional DA Signal Features vs Time Measures — {signal_features_all["session_id"].nunique()} sessions pooled ({CHANNEL_LABEL})', fontsize=14, fontweight='bold')

remaining_features = features[3:]  # sig_at_decision, sig_at_cue, signal_range

for idx, feature in enumerate(remaining_features):
    ax = axes[0, idx]
    clean_data = signal_features_all[[feature, 'time_waited']].dropna()
    if len(clean_data) > 0:
        ax.scatter(clean_data[feature], clean_data['time_waited'], alpha=0.2, s=15)
        if not np.isnan(corr_time_waited[feature]['r']):
            z = np.polyfit(clean_data[feature], clean_data['time_waited'], 1)
            x_line = np.linspace(clean_data[feature].min(), clean_data[feature].max(), 100)
            ax.plot(x_line, np.poly1d(z)(x_line), "r--", alpha=0.8, linewidth=2)
            r, pval = corr_time_waited[feature]['r'], corr_time_waited[feature]['p']
            ax.set_title(f'{feature}\nr={r:.3f}, p={pval:.4f}', fontsize=10)
    ax.set_xlabel(feature)
    ax.set_ylabel('Time Waited (s)')
    ax.grid(alpha=0.3)

for idx, feature in enumerate(remaining_features):
    ax = axes[1, idx]
    clean_data = signal_features_all[[feature, 'decision_time']].dropna()
    if len(clean_data) > 0:
        ax.scatter(clean_data[feature], clean_data['decision_time'], alpha=0.2, s=15, color='orange')
        if not np.isnan(corr_decision_time[feature]['r']):
            z = np.polyfit(clean_data[feature], clean_data['decision_time'], 1)
            x_line = np.linspace(clean_data[feature].min(), clean_data[feature].max(), 100)
            ax.plot(x_line, np.poly1d(z)(x_line), "r--", alpha=0.8, linewidth=2)
            r, pval = corr_decision_time[feature]['r'], corr_decision_time[feature]['p']
            ax.set_title(f'{feature}\nr={r:.3f}, p={pval:.4f}', fontsize=10)
    ax.set_xlabel(feature)
    ax.set_ylabel('Decision Time (s)')
    ax.grid(alpha=0.3)

for idx in range(len(remaining_features), 3):
    axes[0, idx].axis('off')
    axes[1, idx].axis('off')

plt.tight_layout()
plt.show()

# pilot

In [ ]:
session_info = merged_log.iloc[0]
session_id = str(session_info["session_id"])
sig_path = PROCESSED_OUT / session_id / "photometry_with_trial_data.csv"
sig = pd.read_csv(sig_path, dtype={"previous_trial_miss_trial": "object"})

In [ ]:
lr_path = PROCESSED_OUT / session_id / GRAB_RESULTS_FILENAME
lr = pd.read_csv(lr_path)

In [ ]:
behav_subdir = str(session_info["dir"])
trials_path = BEHAV_DIR / behav_subdir / f"trials_analyzed_{behav_subdir}.csv"
trials = pd.read_csv(trials_path, index_col=0)

In [ ]:
# Merge with trials to bring in good_trial and time_waited
trials_with_slope = trials.merge(
    lr[['trial_num', 'slope', 'intercept', 'r_squared', 'p_value', 'std_err']],
    left_on='session_trial_num',
    right_on='trial_num',
    how='left'
)
trials_with_slope['decision_time'] = trials_with_slope['bg_length'] + trials_with_slope['time_waited']

In [ ]:
# Get top 5 best fitted trials (highest r_squared)
non_impulsive_trials = trials_with_slope.loc[trials_with_slope['time_waited'] > NON_IMPULSIVE_CUTOFF]
top_50_fits = non_impulsive_trials.dropna(subset=['r_squared']).nlargest(50, 'r_squared')
top_50_fits[['session_trial_num', 'slope', 'r_squared', 'p_value', 'time_waited', 'decision_time', 'reward']]

In [ ]:
# Display the top 5 best fitted trial plots
from PIL import Image

plot_dir = OUT_PLOTS_DIR / f"{session_info['mouse']}_{session_info['date']}_linear_fits"

fig, axes = plt.subplots(5, 1, figsize=(14, 30))

for idx, (_, trial_row) in enumerate(top_50_fits[:5].iterrows()):
    trial_num = int(trial_row['session_trial_num'])
    plot_path = plot_dir / f"trial_{trial_num:03d}_linear_fit.png"
    
    if plot_path.exists():
        img = Image.open(plot_path)
        axes[idx].imshow(img)
        axes[idx].axis('off')
        axes[idx].set_title(f"Trial {trial_num}: R²={trial_row['r_squared']:.4f}, slope={trial_row['slope']:.4f}", 
                           fontsize=12, pad=10)
    else:
        axes[idx].text(0.5, 0.5, f"Plot not found:\n{plot_path}", 
                      ha='center', va='center', fontsize=10)
        axes[idx].axis('off')

plt.tight_layout()
plt.show()

# Lab meeting plots

In [ ]:
# Build a combined trial-level table across sessions
all_trials = []

for _, session_info in merged_log.iterrows():
    session_id = str(session_info["session_id"])
    lr_path = PROCESSED_OUT / session_id / GRAB_RESULTS_FILENAME
    trials_path = BEHAV_DIR / str(session_info["dir"]) / f"trials_analyzed_{session_info['dir']}.csv"

    if not lr_path.exists() or not trials_path.exists():
        continue

    lr = pd.read_csv(lr_path)
    if lr.empty:
        continue
    trials = pd.read_csv(trials_path, index_col=0)

    trials_with_slope = trials.merge(
        lr[["trial_num", "slope", "intercept", "r_squared", "p_value", "std_err"]],
        left_on="session_trial_num",
        right_on="trial_num",
        how="left",
    )
    trials_with_slope["decision_time"] = (
        trials_with_slope["bg_length"] + trials_with_slope["time_waited"]
    )
    trials_with_slope["session_id"] = session_id
    trials_with_slope["mouse"] = session_info.get("mouse", "")
    trials_with_slope["date"] = session_info.get("date", "")

    all_trials.append(trials_with_slope)

all_trials_df = pd.concat(all_trials, ignore_index=True) if all_trials else pd.DataFrame()

In [ ]:
# -----------------------------
# 1) Scatter: slope vs time
# -----------------------------
if not all_trials_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=False, sharey=True)

    # slope vs decision_time
    x = all_trials_df["slope"].values
    y = all_trials_df["decision_time"].values
    reward = all_trials_df.get("reward", pd.Series([np.nan] * len(all_trials_df)))

    axes[0].scatter(x, y, s=18, alpha=0.35, c=reward, cmap="viridis")
    axes[0].set_xlabel("Ramp slope")
    axes[0].set_ylabel("Decision time (s)")
    axes[0].set_title("Slope vs Decision Time")

    # slope vs time_waited
    y2 = all_trials_df["time_waited"].values
    axes[1].scatter(x, y2, s=18, alpha=0.35, c=reward, cmap="viridis")
    axes[1].set_xlabel("Ramp slope")
    axes[1].set_ylabel("Time waited (s)")
    axes[1].set_title("Slope vs Time Waited")

    plt.tight_layout()
    plt.show()

In [ ]:
# -----------------------------
# 2) Session summary plot
# -----------------------------
if "summary_df" in globals() and not summary_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

    for ax, time_col in zip(axes, ["decision_time", "time_waited"]):
        sub = summary_df[summary_df["time_correlated"] == time_col].copy()
        # sort by mouse/date for tidy x-axis
        sub = sub.sort_values(["mouse", "date", "trial_group"])
        x_labels = sub["mouse"].astype(str) + "_" + sub["date"].astype(str)

        # jitter by trial group for visibility
        trial_groups = sub["trial_group"].unique().tolist()
        offsets = {g: i for i, g in enumerate(trial_groups)}
        x = np.arange(len(sub)) + sub["trial_group"].map(offsets) * 0.12

        ax.errorbar(
            x,
            sub["coef"],
            yerr=sub["stderr"],
            fmt="o",
            alpha=0.7,
            capsize=3,
        )
        ax.axhline(0, color="gray", linewidth=1, alpha=0.6)
        ax.set_title(f"Session regression coef: slope → {time_col}")
        ax.set_xlabel("Session (mouse_date; jittered by group)")
        ax.set_ylabel("Regression coef")
        ax.set_xticks([])

    plt.tight_layout()
    plt.show()

In [ ]:
# -----------------------------
# 3) R² distribution
# -----------------------------
if not all_trials_df.empty:
    fig, ax = plt.subplots(figsize=(7, 4))
    r2 = all_trials_df["r_squared"].dropna()
    ax.hist(r2, bins=30, color="#4c72b0", alpha=0.8)
    ax.set_xlabel("R²")
    ax.set_ylabel("Count")
    ax.set_title("Distribution of trial-wise R²")
    plt.tight_layout()
    plt.show()

In [ ]:
# -----------------------------
# 4) Slope distributions by group
# -----------------------------
if not all_trials_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

    # impulsive vs non-impulsive
    impulsive = all_trials_df[all_trials_df["time_waited"] <= NON_IMPULSIVE_CUTOFF]
    non_impulsive = all_trials_df[all_trials_df["time_waited"] > NON_IMPULSIVE_CUTOFF]

    axes[0].boxplot(
        [impulsive["slope"].dropna(), non_impulsive["slope"].dropna()],
        labels=["impulsive", "non-impulsive"],
        showfliers=False,
    )
    axes[0].set_title("Slope by impulsivity")
    axes[0].set_ylabel("Ramp slope")

    # reward vs no reward (if present)
    if "reward" in all_trials_df.columns:
        reward_yes = all_trials_df[all_trials_df["reward"] == 1]
        reward_no = all_trials_df[all_trials_df["reward"] == 0]
        axes[1].boxplot(
            [reward_yes["slope"].dropna(), reward_no["slope"].dropna()],
            labels=["reward", "no reward"],
            showfliers=False,
        )
        axes[1].set_title("Slope by reward")
    else:
        axes[1].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# -----------------------------
# 5) Within-session timeline
# -----------------------------
if not all_trials_df.empty:
    # Pick a single session (first) for timeline display
    first_session = all_trials_df["session_id"].iloc[0]
    ses = all_trials_df[all_trials_df["session_id"] == first_session]

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(
        ses["session_trial_num"],
        ses["slope"],
        marker="o",
        linestyle="-",
        linewidth=1,
        alpha=0.7,
    )
    ax.set_title(f"Within-session slope timeline: {first_session}")
    ax.set_xlabel("Trial number")
    ax.set_ylabel("Ramp slope")
    ax.axhline(0, color="gray", linewidth=1, alpha=0.6)
    plt.tight_layout()
    plt.show()

In [ ]:
# -----------------------------
# 6) Per-fit scatter (summary_df)
# -----------------------------
if "summary_df" in globals() and not summary_df.empty and not all_trials_df.empty:
    out_dir = OUT_PLOTS_DIR / "session_summary_scatter"
    out_dir.mkdir(parents=True, exist_ok=True)

    def subset_by_group(df: pd.DataFrame, label: str) -> pd.DataFrame:
        if label == "all":
            return df
        if label == "non_impulsive":
            return df[df["time_waited"] > NON_IMPULSIVE_CUTOFF]
        if label == "good":
            if "good_trial" in df.columns:
                return df[df["good_trial"] == True]
            return df
        return df

    def p_to_stars(p_value: float) -> str:
        if not np.isfinite(p_value):
            return "n.s."
        if p_value < 0.001:
            return "***"
        if p_value < 0.01:
            return "**"
        if p_value < 0.05:
            return "*"
        return "n.s."

    for _, row in summary_df.iterrows():
        session_id = row.get("session_id")
        time_col = row.get("time_correlated")
        label = row.get("trial_group")

        if not session_id or not time_col:
            continue

        ses_df = all_trials_df[all_trials_df["session_id"] == session_id].copy()
        ses_df = subset_by_group(ses_df, label)
        plot_df = ses_df.dropna(subset=["slope", time_col])
        if len(plot_df) < 2:
            continue

        x = plot_df["slope"].values
        y = plot_df[time_col].values

        fig, ax = plt.subplots(figsize=(6, 5))
        ax.scatter(x, y, s=20, alpha=0.5, color=PLOT_COLOR)

        coef = row.get("coef", np.nan)
        intercept = row.get("intercept", np.nan)
        if np.isfinite(coef) and np.isfinite(intercept):
            x_line = np.array([np.nanmin(x), np.nanmax(x)])
            y_line = coef * x_line + intercept
            ax.plot(x_line, y_line, color="black", linewidth=2, label="Fit")

        r2 = row.get("r2", np.nan)
        p_val = row.get("p", np.nan)
        n = row.get("n", len(plot_df))
        sig = p_to_stars(p_val)

        ax.set_title(f"{session_id} | {label} | slope → {time_col}")
        ax.set_xlabel("Ramp slope")
        ylabel = "Decision time (s)" if time_col == "decision_time" else "Time waited (s)"
        ax.set_ylabel(ylabel)

        ax.text(
            0.02,
            0.98,
            f"n={int(n)}\nR²={r2:.3f}\np={p_val:.3g} {sig}",
            transform=ax.transAxes,
            va="top",
            ha="left",
            fontsize=10,
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7),
        )
        ax.grid(alpha=0.2)
        plt.tight_layout()

        safe_session = str(session_id).replace("/", "-")
        fname = f"{safe_session}_{label}_{time_col}.png"
        save_path = out_dir / fname
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.close(fig)

    print(f"Saved per-fit scatter plots to: {out_dir}")
else:
    print("summary_df or all_trials_df is missing/empty. Run earlier cells first.")